# TechStore Returns: evidencias, verificación de rúbrica y capturas

**Antes de empezar**
1. En Colab, abre el panel de **Secrets** (icono de llave) y agrega `GOOGLE_API_KEY` con tu API key de AI Studio. Activa el acceso del notebook.
2. Cambia `REPO_URL` y `APP` en la celda de configuración.
3. Ejecuta las celdas **en orden**. Al final se descarga un ZIP con las capturas y la evidencia.

Las capturas salen en la carpeta `capturas/`:
`01_caso_estandar`, `02_fuera_plazo`, `03_alto_valor` (interfaz Streamlit) y tablas de resultados.

In [ ]:
# 1. Configuración (cambiar estos valores)
REPO_URL = "https://github.com/robustam/evaluaci-n2ai.git"  # <- URL real del repositorio
APP = "app.py"                                                      # <- archivo principal de Streamlit
RAIZ = "/content/techstore-returns-agent"

In [ ]:
# 2. Clonar el repositorio
import os
if not os.path.exists(RAIZ):
    !git clone {REPO_URL} {RAIZ}
%cd {RAIZ}
!git log --oneline -3

In [ ]:
# 3. Instalar dependencias (Colab ya trae parte; los avisos de conflicto se pueden ignorar)
!pip install -q -r requirements.txt
!pip install -q playwright pytest matplotlib
!playwright install chromium --with-deps

In [ ]:
# 4. Clave de API desde Secrets de Colab (nunca escribir la clave en el código)
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ.setdefault("LLM_MODEL", "gemini-3.8-flash")

In [ ]:
# 5. Datos ficticios, base de datos e indexación semántica (RAG)
!python scripts/generate_mock_data.py
!python -m scripts.seed_data
!python -m scripts.index_politicas

In [ ]:
# 6. Verificar conexión con Gemini
!python scripts/verify_llm.py

In [ ]:
# 7. Verificador de rúbrica (revisa el código, no reemplaza la evaluación)
import pathlib, re

def buscar(patron, archivos):
    hallazgos = []
    for f in archivos:
        p = pathlib.Path(f)
        if p.is_file() and re.search(patron, p.read_text(encoding="utf-8", errors="ignore"), re.M):
            hallazgos.append(str(p))
    return hallazgos

py_agente = [str(p) for p in pathlib.Path("agent").rglob("*.py")]
checks = {
    "IE1 AgentExecutor con Gemini (crear_agente)": buscar(r"crear_agente|AgentExecutor", py_agente),
    "IE2 Frameworks (LangChain, Chroma, Streamlit)": buscar(r"langchain|chromadb|streamlit", py_agente),
    "IE3 Memoria de corto plazo (ConversationBufferMemory)": buscar(r"ConversationBufferMemory", py_agente),
    "IE3 Memoria de largo plazo (guardar_decision)": buscar(r"guardar_decision", py_agente),
    "IE4 RAG usado dentro del flujo (buscar_casos_similares)": buscar(r"buscar_casos_similares", ["agent/core.py"]),
    "IE5 Planner con reglas (decidir_accion)": buscar(r"decidir_accion", py_agente),
    "IE6/IE8 Diagrama de orquestación en el repo": [str(p) for p in pathlib.Path(".").rglob("*")
                                                     if p.suffix in (".md", ".png", ".svg") and "diagrama" in p.name.lower()],
    "IE8 README con instalación": buscar(r"[Ii]nstalaci", ["README.md"]),
    "IE2/IE8 Versiones exactas (requirements-lock.txt o ==)": (
        ["requirements-lock.txt"] if pathlib.Path("requirements-lock.txt").exists()
        else buscar(r"==", ["requirements.txt"])),
}
for criterio, hallazgos in checks.items():
    print(("✅" if hallazgos else "❌"), criterio, hallazgos if hallazgos else "")

In [ ]:
# 8. Pruebas automatizadas y generación de evidencias
!python -m scripts.generar_evidencias

In [ ]:
# 9. Resumen de la evidencia
print(pathlib.Path("evidence/resumen_evidencias.md").read_text(encoding="utf-8"))

In [ ]:
# 10. Tablas de evidencia en PNG (capturas para el informe)
import json, textwrap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

CAPTURAS = Path("capturas")
CAPTURAS.mkdir(exist_ok=True)
esc = json.loads(Path("evidence/escenarios.json").read_text(encoding="utf-8"))

def guardar_tabla(filas, columnas, titulo, ruta, ancho=13):
    fig, ax = plt.subplots(figsize=(ancho, 0.8 + 0.5 * len(filas)))
    ax.axis("off")
    tabla = ax.table(cellText=filas, colLabels=columnas, loc="center", cellLoc="left")
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(10)
    tabla.scale(1, 1.6)
    ax.set_title(titulo, loc="left", fontsize=13, weight="bold")
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)

def guardar_texto(lineas, titulo, ruta):
    fig, ax = plt.subplots(figsize=(13, 0.22 * len(lineas) + 1))
    ax.axis("off")
    ax.text(0, 1, "\n".join(lineas), family="monospace", fontsize=9, va="top")
    ax.set_title(titulo, loc="left", fontsize=12, weight="bold")
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)

# Resumen de los 6 escenarios
filas = [[e["nombre"], ", ".join(p.get("pedido_id", "?") for p in e["pasos"]),
          " → ".join(e["esperado"]), " → ".join(e["obtenido"]), "OK" if e["ok"] else "FALLA"] for e in esc]
guardar_tabla(filas, ["Escenario", "Pedido(s)", "Esperado", "Obtenido", "Resultado"],
              "Resumen de escenarios del agente TechStore Returns", "capturas/00_resumen_escenarios.png")

# Traza de cada escenario (decisión, razón y tools en orden)
nombres_archivo = ["01_caso_estandar", "02_fuera_plazo", "03_alto_valor",
                   "04_memoria_largo_plazo", "05_sospecha_fraude", "06_falta_informacion"]
for e, nombre in zip(esc, nombres_archivo):
    filas = []
    for p in e["pasos"]:
        if p.get("ok"):
            filas.append([p["pedido_id"], p.get("accion", "?"),
                          textwrap.fill(p.get("razon", ""), 60),
                          textwrap.fill(" → ".join(p.get("tools_ejecutadas", [])), 60)])
        else:
            filas.append([p.get("pedido_id", "?"), "ERROR", textwrap.fill(str(p.get("error")), 60), ""])
    guardar_tabla(filas, ["Pedido", "Decisión", "Razón", "Tools ejecutadas (en orden)"], e["nombre"],
                  f"capturas/{nombre}_traza.png", ancho=14)

# Salida de pytest
lineas_pytest = Path("evidence/pytest_output.txt").read_text(encoding="utf-8").splitlines()
guardar_texto(lineas_pytest[-60:], "Salida de pytest (últimas líneas)", "capturas/07_pytest.png")
print("Capturas generadas:", sorted(p.name for p in CAPTURAS.glob("*.png")))

In [ ]:
# 11. Capturas de la interfaz Streamlit (Playwright dentro de Colab)
import subprocess, sys, time
from playwright.sync_api import sync_playwright

PUERTO = 8501
log = open("capturas/streamlit.log", "w")
servidor = subprocess.Popen([sys.executable, "-m", "streamlit", "run", APP,
                             "--server.headless", "true", "--server.port", str(PUERTO)],
                            stdout=log, stderr=log)
time.sleep(10)

# Pedido -> nombre del archivo de captura
PEDIDOS_CAPTURA = [("PED-001", "01_caso_estandar"),
                   ("PED-015", "02_fuera_plazo"),
                   ("PED-022", "03_alto_valor")]

try:
    with sync_playwright() as p:
        navegador = p.chromium.launch(args=["--no-sandbox"])
        pagina = navegador.new_page(viewport={"width": 1280, "height": 900})
        pagina.goto(f"http://localhost:{PUERTO}", wait_until="networkidle")
        for pid, nombre in PEDIDOS_CAPTURA:
            # Ajustar si la interfaz usa otro control (por ejemplo un botón "Procesar").
            caja = pagina.get_by_role("textbox").first
            caja.fill(pid)
            caja.press("Enter")
            pagina.wait_for_timeout(8000)
            pagina.screenshot(path=f"capturas/{nombre}_interfaz.png", full_page=True)
            print("Captura:", nombre)
        navegador.close()
finally:
    servidor.terminate()
    log.close()

In [ ]:
# 12. Vista previa de todas las capturas
from IPython.display import display, Image
import glob
for ruta in sorted(glob.glob("capturas/*.png")):
    print(ruta)
    display(Image(ruta, width=900))

In [ ]:
# 13. Descargar capturas y evidencia en un ZIP
import shutil
from google.colab import files
shutil.make_archive("techstore_capturas_evidencia", "zip", root_dir=".", base_dir="capturas")
shutil.make_archive("techstore_evidence", "zip", root_dir=".", base_dir="evidence")
files.download("techstore_capturas_evidencia.zip")
files.download("techstore_evidence.zip")